# STEP 3 — CVD / ALD 판별 헤드
현재 evaluate와 같은 API를 사용합니다. 기본 실행은 저장된 헤드 로딩이며 재학습하지 않습니다. 과거 구현은 `originals/`에 보존했습니다.


In [ ]:
# 처음 한 번 PROJECT_PATH에 프로젝트 루트를 지정합니다. None은 저장된 설정 / 자동 탐색.
PROJECT_PATH = None  # 예: r"D:\GEVIS"
from pathlib import Path
import runpy
_source_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'processing/workflow.py').is_file())
workflow = runpy.run_path(str(_source_root / 'processing/workflow.py'))
ROOT = PROJECT_ROOT = workflow['configure'](PROJECT_PATH)


In [ ]:
DOMAIN = 'PECVD'  # PECVD 또는 PEALD
ACTION = 'load'    # load: 저장본 재사용 / train: 전처리·헤드 학습·평가 실행
CHECKPOINTS = None # None: 도메인별 저장본 탐색. 또는 {'softmax': 'artifacts/.../answer_head.pt'}
ENCODER = 'DEP.'


In [ ]:
import pandas as pd
from pecvd.gevis import load_gevis
from pecvd import gevis_training as training
GEVIS = load_gevis(ROOT)
display(GEVIS.describe())


In [ ]:
if DOMAIN not in ('PECVD', 'PEALD') or ACTION not in ('load', 'train'):
    raise ValueError('DOMAIN / ACTION 설정을 확인하세요.')
if ACTION == 'train':
    if DOMAIN == 'PECVD':
        TRAINING = training.TrainingConfig(epochs=80, patience=12, batch_size=256, max_seconds=600.)
        PREPARED = training.prepare_cvd(ROOT, GEVIS)
        RESULTS = training.train_cvd(ROOT, GEVIS, PREPARED, training=TRAINING)
    else:
        PREPROCESSING = training.PreprocessingConfig(schema_columns=None, max_window_length=64,
            encode_valves=True, drop_columns=(), context_columns=())
        NATIVE = training.NativeConfig(excluded_months=(), recipe_suffix='')
        TRAINING = training.TrainingConfig(epochs=40, patience=8, batch_size=256, max_seconds=600.)
        CHOICES = [training.ChoiceConfig(mode=mode, groups=((1,3),(2,),(4,),(5,)), sigmoid_threshold=.5)
                   for mode in ('softmax', 'grouped', 'sigmoid')]
        PREPARED = training.prepare_ald(ROOT, GEVIS, encoder=ENCODER,
            preprocessing=PREPROCESSING, native=NATIVE, max_memory_mb=4096)
        RESULTS = training.train_ald(ROOT, GEVIS, PREPARED, encoder=ENCODER,
            preprocessing=PREPROCESSING, native=NATIVE, training=TRAINING, choices=CHOICES)
    display(RESULTS.summary)
    CHECKPOINTS = RESULTS.checkpoints
RUNTIMES, SAVED_HEADS = workflow['load_heads'](ROOT, GEVIS, DOMAIN, CHECKPOINTS)
display(SAVED_HEADS)
print('다음: STEP_3.1에서 같은 DOMAIN을 선택합니다. 저장본 로딩에는 재학습이 필요하지 않습니다.')
